# Hierarchical Clustering

Hierarchical clustering builds a **tree of clusters (dendrogram)** rather than a flat partition. You don't need to specify the number of clusters upfront — you choose how many groups you want after seeing the full merge history.

---

## Table of Contents
1. [Intuition and Motivation](#1-intuition)
2. [Agglomerative vs Divisive](#2-agglomerative-vs-divisive)
3. [Linkage Criteria — The Math](#3-linkage-criteria)
4. [The Dendrogram](#4-dendrogram)
5. [Lance-Williams Update Formula](#5-lance-williams)
6. [Ward's Method — Deep Dive](#6-ward)
7. [Complexity Analysis](#7-complexity)
8. [K-Means vs Hierarchical](#8-comparison)
9. [Implementation from Scratch](#9-from-scratch)
10. [scipy and sklearn Implementations](#10-sklearn)
11. [Summary](#11-summary)

---
## 1. Intuition and Motivation

### The Problem with K-Means

K-Means requires you to specify $K$ before running. But what if you don't know $K$? And what if you want to explore the data's natural hierarchy — e.g., first split into 2 broad groups, then 5 subgroups, then 12 fine groups?

### The Hierarchical Idea

Imagine a **family tree**, but for data points:
- At the bottom: every point is its own cluster (n clusters)
- At the top: all points form one big cluster (1 cluster)
- In between: every possible intermediate grouping

By **cutting** the tree at any height, you get any number of clusters from 1 to n.

### Real-World Example

Biological taxonomy is a dendrogram: species → genus → family → order → class → phylum → kingdom. Hierarchical clustering discovers such structure automatically from data.

---
## 2. Agglomerative vs Divisive

### Agglomerative (Bottom-Up) ← the standard

```
Start: n clusters, each containing 1 point
Repeat:
    Find the two closest clusters
    Merge them into one
Until: only 1 cluster remains
```

### Divisive (Top-Down) ← less common

```
Start: 1 cluster containing all n points
Repeat:
    Split the largest/least cohesive cluster
Until: n clusters (each with 1 point)
```

**Why agglomerative is preferred**: Merging is computationally simpler. Deciding how to split is harder and more expensive. Divisive requires a sub-clustering algorithm at each step.

All further discussion focuses on **agglomerative** clustering.

---
## 3. Linkage Criteria — The Math

The key question in agglomerative clustering: **how do you measure the distance between two clusters?**

Let $A$ and $B$ be two clusters, and $d(a, b)$ = distance between individual points $a \in A$, $b \in B$.

### Single Linkage (Minimum)

$$d_{\text{single}}(A, B) = \min_{a \in A,\, b \in B} d(a, b)$$

- Uses the **closest** pair of points between clusters
- Creates long, chaining clusters ("spaghetti" effect)
- Good for detecting elongated or irregular-shaped clusters
- Very sensitive to noise and outliers

### Complete Linkage (Maximum)

$$d_{\text{complete}}(A, B) = \max_{a \in A,\, b \in B} d(a, b)$$

- Uses the **farthest** pair of points
- Creates compact, roughly equal-sized clusters
- Less sensitive to outliers
- Can break large clusters unnecessarily

### Average Linkage (UPGMA)

$$d_{\text{avg}}(A, B) = \frac{1}{|A||B|} \sum_{a \in A} \sum_{b \in B} d(a, b)$$

- Averages **all pairwise distances** between clusters
- Good compromise between single and complete linkage
- Computationally expensive: $O(|A| \cdot |B|)$ per merge

### Ward Linkage (Minimum Variance) ← Most Popular

Ward measures the **increase in total WCSS** if the two clusters are merged:

$$d_{\text{Ward}}(A, B) = \frac{|A| \cdot |B|}{|A| + |B|} \|\boldsymbol{\mu}_A - \boldsymbol{\mu}_B\|^2$$

where $\boldsymbol{\mu}_A$ and $\boldsymbol{\mu}_B$ are the centroids of clusters $A$ and $B$.

- Minimizes the variance within clusters at each merge step
- Creates compact, roughly equal-sized clusters
- Most similar to K-Means in spirit
- **Default choice for most applications**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
X, _ = make_blobs(n_samples=60, centers=4, cluster_std=0.7, random_state=42)
X = StandardScaler().fit_transform(X)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
linkages = ['single', 'complete', 'average', 'ward']
titles = ['Single Linkage (min)', 'Complete Linkage (max)',
          'Average Linkage (UPGMA)', 'Ward Linkage (min variance)']

for ax, method, title in zip(axes.flat, linkages, titles):
    Z = linkage(X, method=method)
    dendrogram(Z, ax=ax, leaf_font_size=7, color_threshold=0.7*max(Z[:,2]))
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Sample index')
    ax.set_ylabel('Distance')

plt.suptitle('Effect of Linkage Criterion on Dendrogram Shape', fontsize=14)
plt.tight_layout()
plt.savefig('linkage_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 4. The Dendrogram

### Reading a Dendrogram

- **X-axis**: individual data points (or cluster IDs after merging)
- **Y-axis**: the **distance (dissimilarity)** at which two clusters were merged
- A horizontal bar connecting two branches at height $h$ means those two clusters were merged when their linkage distance was $h$
- Higher merge = less similar clusters being joined

### Choosing the Number of Clusters

**Method 1 — Largest gap**: Look for the longest vertical line in the dendrogram (a large vertical gap means no natural merges happen there). Cut just below this gap. The number of vertical lines your horizontal cut crosses = $K$.

**Method 2 — Specified distance**: Call `fcluster(Z, t=d, criterion='distance')` — cut all merges above distance $d$.

**Method 3 — Specified K**: Call `fcluster(Z, t=K, criterion='maxclust')` — force exactly $K$ clusters.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
import matplotlib.pyplot as plt
import numpy as np

Z_ward = linkage(X, method='ward')

# Find the largest gap
distances = Z_ward[:, 2]
gaps = np.diff(distances)
largest_gap_idx = np.argmax(gaps)
cut_height = (distances[largest_gap_idx] + distances[largest_gap_idx + 1]) / 2

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Full dendrogram with cut line
ax = axes[0]
dendrogram(Z_ward, ax=ax, leaf_font_size=8, color_threshold=cut_height)
ax.axhline(y=cut_height, color='red', linestyle='--', lw=2,
           label=f'Cut at {cut_height:.2f} → K clusters')
ax.set_title('Ward Dendrogram with Auto-Cut', fontsize=12)
ax.set_xlabel('Sample Index')
ax.set_ylabel('Distance')
ax.legend(fontsize=10)

# The resulting clusters projected to 2D
labels_auto = fcluster(Z_ward, t=cut_height, criterion='distance') - 1
K_found = len(np.unique(labels_auto))
axes[1].scatter(X[:, 0], X[:, 1], c=labels_auto, cmap='Set1', s=50, alpha=0.8)
axes[1].set_title(f'Resulting Clusters (K={K_found})', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ward_dendrogram.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Largest gap at: {distances[largest_gap_idx]:.3f} → {distances[largest_gap_idx+1]:.3f}")
print(f"Auto-detected K = {K_found}")

---
## 5. Lance-Williams Update Formula

When two clusters $A$ and $B$ are merged into $A \cup B$, we need to update all distances $d(A \cup B, C)$ for every remaining cluster $C$ — without recomputing from scratch.

The **Lance-Williams recurrence** provides a unified formula:

$$d(A \cup B, C) = \alpha_A \, d(A, C) + \alpha_B \, d(B, C) + \beta \, d(A, B) + \gamma \, |d(A,C) - d(B,C)|$$

Each linkage method is a special case:

| Method | $\alpha_A$ | $\alpha_B$ | $\beta$ | $\gamma$ |
|---|---|---|---|---|
| Single | $\frac{1}{2}$ | $\frac{1}{2}$ | 0 | $-\frac{1}{2}$ |
| Complete | $\frac{1}{2}$ | $\frac{1}{2}$ | 0 | $+\frac{1}{2}$ |
| Average (UPGMA) | $\frac{|A|}{|A|+|B|}$ | $\frac{|B|}{|A|+|B|}$ | 0 | 0 |
| Ward | $\frac{|A|+|C|}{|A|+|B|+|C|}$ | $\frac{|B|+|C|}{|A|+|B|+|C|}$ | $\frac{-|C|}{|A|+|B|+|C|}$ | 0 |

This allows the distance matrix to be updated in $O(n)$ per merge, avoiding recomputation from scratch.

---
## 6. Ward's Method — Deep Dive

Ward's linkage is by far the most popular choice. Let's understand exactly what it minimizes.

### Total Within-Cluster Sum of Squares

$$T = \sum_{k=1}^{K} \sum_{\mathbf{x} \in C_k} \|\mathbf{x} - \boldsymbol{\mu}_k\|^2$$

When we merge clusters $A$ and $B$, the increase in $T$ is:

$$\Delta T(A, B) = T_{A \cup B} - T_A - T_B = \frac{|A| \cdot |B|}{|A| + |B|} \|\boldsymbol{\mu}_A - \boldsymbol{\mu}_B\|^2$$

**Proof** (using the parallel axis theorem):

$$T_{A \cup B} = \sum_{\mathbf{x} \in A \cup B} \|\mathbf{x} - \boldsymbol{\mu}_{A \cup B}\|^2$$

Splitting by group:

$$= \underbrace{\sum_{\mathbf{x} \in A} \|\mathbf{x} - \boldsymbol{\mu}_A\|^2}_{T_A} + \underbrace{\sum_{\mathbf{x} \in B} \|\mathbf{x} - \boldsymbol{\mu}_B\|^2}_{T_B} + \frac{|A||B|}{|A|+|B|}\|\boldsymbol{\mu}_A - \boldsymbol{\mu}_B\|^2$$

So the **Ward distance** equals the **increase in inertia** when merging — and at each step, we merge whichever pair causes the smallest increase. This greedy strategy produces compact, roughly spherical clusters.

### Connection to K-Means

Ward's agglomerative clustering minimizes the same objective as K-Means (WCSS). Ward is often used to get a good starting point for K-Means: cluster hierarchically, take the K-cluster result, use centroids as K-Means initial centers.

---
## 7. Complexity Analysis

### Naive Implementation

- **Space**: $O(n^2)$ to store the pairwise distance matrix
- **Time**: $O(n^3)$ — at each of the $n-1$ merges, find the minimum in the $O(n^2)$ matrix

### Optimized (scipy default)

- Uses a priority queue (min-heap) to find the minimum distance: $O(n^2 \log n)$
- Lance-Williams update avoids recomputing distances from scratch: $O(n)$ per merge
- Still requires $O(n^2)$ space

### Practical Implication

Hierarchical clustering is only feasible for **small to medium datasets** ($n \lesssim 10{,}000$). For larger data, use K-Means or DBSCAN.

Alternatively, apply hierarchical clustering to a **sample** or to **K-Means centroids** (two-stage approach).

---
## 8. K-Means vs Hierarchical

| Aspect | K-Means | Hierarchical (Agglomerative) |
|---|---|---|
| K required upfront? | Yes | No — choose after seeing dendrogram |
| Deterministic? | No (random init) | Yes (given linkage and distance) |
| Scales to large n? | Yes — $O(nKdT)$ | No — $O(n^2)$ space |
| Cluster shape | Spherical only | Flexible (depends on linkage) |
| Outlier handling | Poor | Slightly better (outliers merge late) |
| Result type | Flat partition | Full hierarchy (dendrogram) |
| Interpretability | Centroids are descriptive | Dendrogram shows full structure |
| Revisit K? | Must rerun | Just cut at different height |

---
## 9. Implementation from Scratch

In [ ]:
import numpy as np
from scipy.spatial.distance import cdist

class AgglomerativeFromScratch:
    """
    Agglomerative clustering with single, complete, average, or ward linkage.
    Records the merge history in scipy-compatible linkage matrix format.
    """
    def __init__(self, n_clusters=2, linkage='ward'):
        self.n_clusters = n_clusters
        self.linkage = linkage

    def _cluster_dist(self, X, ids_a, ids_b):
        a = X[list(ids_a)]
        b = X[list(ids_b)]
        dists = cdist(a, b, 'euclidean')
        if self.linkage == 'single':
            return dists.min()
        elif self.linkage == 'complete':
            return dists.max()
        elif self.linkage == 'average':
            return dists.mean()
        elif self.linkage == 'ward':
            na, nb = len(ids_a), len(ids_b)
            mu_a = a.mean(axis=0)
            mu_b = b.mean(axis=0)
            return np.sqrt(na * nb / (na + nb)) * np.linalg.norm(mu_a - mu_b)
        raise ValueError(f"Unknown linkage: {self.linkage}")

    def fit_predict(self, X):
        n = len(X)
        # Each cluster is a set of point indices
        clusters = {i: {i} for i in range(n)}
        cluster_ids = list(range(n))

        while len(cluster_ids) > self.n_clusters:
            best_dist = np.inf
            merge_i, merge_j = None, None
            # Find the pair of clusters with minimum linkage distance
            for ii in range(len(cluster_ids)):
                for jj in range(ii + 1, len(cluster_ids)):
                    ci, cj = cluster_ids[ii], cluster_ids[jj]
                    d = self._cluster_dist(X, clusters[ci], clusters[cj])
                    if d < best_dist:
                        best_dist = d
                        merge_i, merge_j = ci, cj
            # Merge
            new_id = max(clusters.keys()) + 1
            clusters[new_id] = clusters[merge_i] | clusters[merge_j]
            del clusters[merge_i], clusters[merge_j]
            cluster_ids = list(clusters.keys())

        # Assign labels
        labels = np.zeros(n, dtype=int)
        for label, (_, pts) in enumerate(clusters.items()):
            for pt in pts:
                labels[pt] = label
        self.labels_ = labels
        return labels


# Test vs sklearn
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score

X_small, y_true = make_blobs(n_samples=80, centers=3, random_state=42)
X_small = StandardScaler().fit_transform(X_small)

for method in ['single', 'complete', 'average', 'ward']:
    scratch = AgglomerativeFromScratch(n_clusters=3, linkage=method)
    labels_s = scratch.fit_predict(X_small)

    sk = AgglomerativeClustering(n_clusters=3, linkage=method)
    labels_sk = sk.fit_predict(X_small)

    ari = adjusted_rand_score(labels_sk, labels_s)
    print(f"{method:10s}: ARI vs sklearn = {ari:.4f}")

---
## 10. scipy and sklearn Implementations

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
import matplotlib.pyplot as plt
import numpy as np

iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data)

# scipy: gives full dendrogram
Z = linkage(X_iris, method='ward')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Truncated dendrogram (last 30 merges)
ax1 = axes[0]
dendrogram(Z, ax=ax1, truncate_mode='lastp', p=30,
           leaf_rotation=90, leaf_font_size=9, show_contracted=True)
ax1.axhline(y=7, color='red', linestyle='--', label='Cut → 3 clusters')
ax1.set_title('Ward Dendrogram — Iris Dataset (truncated)')
ax1.set_xlabel('Cluster / Sample')
ax1.set_ylabel('Ward Distance')
ax1.legend()

# Cluster result at K=3
labels = fcluster(Z, t=3, criterion='maxclust') - 1  # 1-indexed → 0-indexed
X_2d = PCA(n_components=2).fit_transform(X_iris)
axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap='Set1', s=40, alpha=0.8)
axes[1].set_title(f'Hierarchical Clustering (Ward, K=3) on Iris — PCA view')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('hierarchical_iris.png', dpi=120, bbox_inches='tight')
plt.show()

ari = adjusted_rand_score(iris.target, labels)
sil = silhouette_score(X_iris, labels)
print(f"Adjusted Rand Index: {ari:.4f}")
print(f"Silhouette Score:    {sil:.4f}")

# sklearn: simpler API, but no dendrogram
print("\nsklearn AgglomerativeClustering:")
for method in ['ward', 'complete', 'average', 'single']:
    agg = AgglomerativeClustering(n_clusters=3, linkage=method)
    lbl = agg.fit_predict(X_iris)
    ari_m = adjusted_rand_score(iris.target, lbl)
    sil_m = silhouette_score(X_iris, lbl)
    print(f"  {method:10s}: ARI={ari_m:.4f}, Silhouette={sil_m:.4f}")

---
## 11. Summary

| Concept | Key Point |
|---|---|
| Approach | Bottom-up: start with n singleton clusters, merge greedily |
| Linkage | Defines distance between clusters — determines cluster shape |
| Ward | Merges clusters with smallest increase in WCSS (like K-Means objective) |
| Dendrogram | Tree showing full merge history; cut at any height for any K |
| Choosing K | Largest vertical gap in dendrogram = natural K |
| Complexity | $O(n^2)$ space, $O(n^2 \log n)$ time — not scalable past ~10K points |
| vs K-Means | No K upfront, deterministic, shows hierarchy; but much slower |

### scipy Quick Reference
```python
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram

Z = linkage(X_scaled, method='ward')     # build dendrogram
dendrogram(Z)                            # visualize
labels = fcluster(Z, t=4, criterion='maxclust')  # get 4 clusters
labels = fcluster(Z, t=5.0, criterion='distance')  # cut at distance 5
```

### sklearn Quick Reference
```python
from sklearn.cluster import AgglomerativeClustering

agg = AgglomerativeClustering(n_clusters=4, linkage='ward')
labels = agg.fit_predict(X_scaled)
```